In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Load the CSV dataset into a pandas DataFrame
df = pd.read_csv('netflix_titles.csv')

# Display the first few rows and column names to understand the structure
display(df.head())
print(df.columns)

# Create a SQLite database engine and export the DataFrame to a table
engine = create_engine('sqlite:///netflix.db')
df.to_sql('netflix', engine, if_exists='replace', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'netflix_titles.csv'

In [ ]:
# Query 1: Select all records from the Netflix table
display(pd.read_sql_query("SELECT * FROM netflix LIMIT 5;", engine))

# Query 2: Select distinct show IDs
display(pd.read_sql_query("SELECT DISTINCT show_id FROM netflix LIMIT 5;", engine))

# Query 3: Select distinct titles of shows released after 2020
display(pd.read_sql_query("SELECT DISTINCT title FROM netflix WHERE release_year > 2020 LIMIT 5;", engine))

# Query 4: Select distinct titles with multiple conditions (AND, OR, NOT)
display(pd.read_sql_query("""
    SELECT DISTINCT title 
    FROM netflix 
    WHERE release_year > 2020 AND type = 'Movie' AND NOT rating = 'R'
    LIMIT 5;
""", engine))

# Query 5: Order titles by release year descending and title ascending
display(pd.read_sql_query("""
    SELECT title, release_year 
    FROM netflix 
    ORDER BY release_year DESC, title ASC 
    LIMIT 5;
""", engine))

In [ ]:
# Query 6: Limit values to the first 5 records
display(pd.read_sql_query("SELECT * FROM netflix LIMIT 5;", engine))

# Query 7: Aggregate functions (Min, Max, Count, Avg, Sum) on release years
display(pd.read_sql_query("""
    SELECT 
        MIN(release_year) AS min_year,
        MAX(release_year) AS max_year,
        COUNT(show_id) AS total_shows,
        AVG(release_year) AS avg_year,
        SUM(release_year) AS sum_year
    FROM netflix;
""", engine))

# Query 8: Use LIKE to find specific country patterns
display(pd.read_sql_query("SELECT DISTINCT country FROM netflix WHERE country LIKE '%United States%' LIMIT 5;", engine))

# Query 9: Use IN operator to filter by a specific list of countries
display(pd.read_sql_query("""
    SELECT title, country 
    FROM netflix 
    WHERE country IN ('United States', 'India', 'United Kingdom') 
    LIMIT 5;
""", engine))

# Query 10: Use BETWEEN to filter records by release year range
display(pd.read_sql_query("SELECT title, release_year FROM netflix WHERE release_year BETWEEN 2020 AND 2021 LIMIT 5;", engine))

In [ ]:
# Query 11: Self-join the Netflix table
display(pd.read_sql_query("""
    SELECT a.title AS title_1, b.title AS title_2, a.release_year 
    FROM netflix a 
    JOIN netflix b ON a.release_year = b.release_year AND a.show_id <> b.show_id 
    LIMIT 5;
""", engine))

# Query 12: Union two different query selections
display(pd.read_sql_query("""
    SELECT title, type FROM netflix WHERE release_year = 2021
    UNION
    SELECT title, type FROM netflix WHERE release_year = 2020
    LIMIT 5;
""", engine))

# Query 13: Use CASE statements for conditional logic (eras)
display(pd.read_sql_query("""
    SELECT title, release_year,
        CASE 
            WHEN release_year >= 2020 THEN 'Recent'
            WHEN release_year >= 2015 THEN 'Modern'
            ELSE 'Classic'
        END AS era
    FROM netflix
    LIMIT 5;
""", engine))

# Query 14: Use a subquery to find shows with the maximum release year
display(pd.read_sql_query("""
    SELECT title, release_year 
    FROM netflix 
    WHERE release_year = (SELECT MAX(release_year) FROM netflix)
    LIMIT 5;
""", engine))

# Query 15: Use COALESCE to handle NULL values in the director column
display(pd.read_sql_query("SELECT title, COALESCE(director, 'Unknown Director') AS director_name FROM netflix LIMIT 5;", engine))

In [ ]:
# Query 16: Convert / Cast release year to text data type
display(pd.read_sql_query("SELECT title, CAST(release_year AS TEXT) AS release_year_str FROM netflix LIMIT 5;", engine))

# Query 17: Window functions LAG and LEAD to inspect adjacent titles
display(pd.read_sql_query("""
    SELECT title, release_year,
        LAG(title, 1) OVER (ORDER BY release_year) AS prev_title,
        LEAD(title, 1) OVER (ORDER BY release_year) AS next_title
    FROM netflix
    LIMIT 5;
""", engine))

# Query 18: Window function ROW_NUMBER partitioned by type
display(pd.read_sql_query("""
    SELECT title, type,
        ROW_NUMBER() OVER (PARTITION BY type ORDER BY release_year DESC) AS row_num
    FROM netflix
    LIMIT 5;
""", engine))

# Query 19: Window function DENSE_RANK ordered by release year
display(pd.read_sql_query("""
    SELECT title, release_year,
        DENSE_RANK() OVER (ORDER BY release_year DESC) AS year_rank
    FROM netflix
    LIMIT 5;
""", engine))

# Query 20: Use WITH clause for a Common Table Expression (CTE)
display(pd.read_sql_query("""
    WITH recent_shows AS (
        SELECT title, release_year, type
        FROM netflix
        WHERE release_year >= 2021
    )
    SELECT type, COUNT(*) AS total_count
    FROM recent_shows
    GROUP BY type;
""", engine))